In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
FEATURES_CSV = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_features.csv"

matches_clean = pd.read_csv(FEATURES_CSV)

print("Loaded successfully!")
print("Shape:", matches_clean.shape)

Loaded successfully!
Shape: (1175, 17)


In [3]:
matches_clean

,match_id,season,match_date,team1,team2,venue,city,toss_winner,toss_decision,toss_winner_won,toss_decision_encoded,team1_win_rate,team2_win_rate,win_rate_diff,winner,result,target
0,335982,2007/08,2008/04/18,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bengaluru,field,0,1,0.49,0.51,-0.02,Kolkata Knight Riders,normal,0
1,335983,2007/08,2008/04/19,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,1,0,0.45,0.56,-0.11,Chennai Super Kings,normal,0
2,335984,2007/08,2008/04/19,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,0,0,0.44,0.49,-0.05,Delhi Capitals,normal,1
3,335986,2007/08,2008/04/20,Kolkata Knight Riders,Sunrisers Hyderabad,Eden Gardens,Kolkata,Sunrisers Hyderabad,bat,0,0,0.51,0.45,0.06,Kolkata Knight Riders,normal,1
4,335985,2007/08,2008/04/20,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai,Mumbai Indians,bat,0,0,0.55,0.49,0.06,Royal Challengers Bengaluru,normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,1527675,2026,2026/03/29,Kolkata Knight Riders,Mumbai Indians,"Wankhede Stadium, Mumbai",Mumbai,Mumbai Indians,field,1,1,0.51,0.55,-0.04,Mumbai Indians,normal,0
1171,1527676,2026,2026/03/30,Chennai Super Kings,Rajasthan Royals,"Barsapara Cricket Stadium, Guwahati",Guwahati,Rajasthan Royals,field,1,1,0.56,0.49,0.07,Rajasthan Royals,normal,0
1172,1527677,2026,2026/03/31,Gujarat Titans,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Punjab Kings,field,1,1,0.61,0.45,0.16,Punjab Kings,normal,0
1173,1527678,2026,2026/04/01,Lucknow Super Giants,Delhi Capitals,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Delhi Capitals,field,1,1,0.51,0.44,0.07,Delhi Capitals,normal,0


In [4]:
X = matches_clean[["toss_winner_won", "toss_decision_encoded",
                    "team1_win_rate", "team2_win_rate", "win_rate_diff"]]
y = matches_clean["target"]
print("Input features shape:", X.shape)
print("Target shape:", y.shape)

Input features shape: (1175, 5)
Target shape: (1175,)


In [5]:
X.isna().sum()

toss_winner_won          0
toss_decision_encoded    0
team1_win_rate           0
team2_win_rate           0
win_rate_diff            0
dtype: int64

In [6]:
y.isna().sum()

np.int64(0)

In [8]:
X = X.dropna()
y = y[X.index]
print("Rows after removing NaN:", len(X))

Rows after removing NaN: 1175


In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,      
    random_state=42     
)

print("Training rows :", len(X_train))
print("Testing rows  :", len(X_test))

Training rows : 822
Testing rows  : 353


# Logistic Regression

In [12]:
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)
print("Logistic Regression Accuracy:", round(lr_accuracy * 100, 2), "%")

Logistic Regression Accuracy: 62.61 %


# Random Forest

In [13]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_preds)
print("Random Forest Accuracy:", round(rf_accuracy * 100, 2), "%")

Random Forest Accuracy: 69.97 %


# XGboost

In [14]:
xgb_model = XGBClassifier(random_state=42, eval_metric="logloss")
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, xgb_preds)
print("XGBoost Accuracy:", round(xgb_accuracy * 100, 2), "%")

XGBoost Accuracy: 72.24 %


# Model Comparasion

In [15]:
results = pd.DataFrame({
    "Model"   : ["Logistic Regression", "Random Forest", "XGBoost"],
    "Accuracy": [round(lr_accuracy * 100, 2),
                 round(rf_accuracy * 100, 2),
                 round(xgb_accuracy * 100, 2)]
})
results = results.sort_values("Accuracy", ascending=False).reset_index(drop=True)
print(results)

                 Model  Accuracy
0              XGBoost     72.24
1        Random Forest     69.97
2  Logistic Regression     62.61


### We got accuracy of 72% using XGBoost

### We will try to see if we can improve the accuracy of this model by doing feature engineering

###### Recent Form

In [16]:
team1_results = matches_clean[["match_id", "match_date", "team1", "winner"]].copy()
team1_results.columns = ["match_id", "match_date", "team", "winner"]
team1_results["won"] = (team1_results["team"] == team1_results["winner"]).astype(int)
team1_results

,match_id,match_date,team,winner,won
0,335982,2008/04/18,Royal Challengers Bengaluru,Kolkata Knight Riders,0
1,335983,2008/04/19,Punjab Kings,Chennai Super Kings,0
2,335984,2008/04/19,Delhi Capitals,Delhi Capitals,1
3,335986,2008/04/20,Kolkata Knight Riders,Kolkata Knight Riders,1
4,335985,2008/04/20,Mumbai Indians,Royal Challengers Bengaluru,0
...,...,...,...,...,...
1170,1527675,2026/03/29,Kolkata Knight Riders,Mumbai Indians,0
1171,1527676,2026/03/30,Chennai Super Kings,Rajasthan Royals,0
1172,1527677,2026/03/31,Gujarat Titans,Punjab Kings,0
1173,1527678,2026/04/01,Lucknow Super Giants,Delhi Capitals,0


In [17]:
team2_results = matches_clean[["match_id", "match_date", "team2", "winner"]].copy()
team2_results.columns = ["match_id", "match_date", "team", "winner"]
team2_results["won"] = (team2_results["team"] == team2_results["winner"]).astype(int)
team2_results

,match_id,match_date,team,winner,won
0,335982,2008/04/18,Kolkata Knight Riders,Kolkata Knight Riders,1
1,335983,2008/04/19,Chennai Super Kings,Chennai Super Kings,1
2,335984,2008/04/19,Rajasthan Royals,Delhi Capitals,0
3,335986,2008/04/20,Sunrisers Hyderabad,Kolkata Knight Riders,0
4,335985,2008/04/20,Royal Challengers Bengaluru,Royal Challengers Bengaluru,1
...,...,...,...,...,...
1170,1527675,2026/03/29,Mumbai Indians,Mumbai Indians,1
1171,1527676,2026/03/30,Rajasthan Royals,Rajasthan Royals,1
1172,1527677,2026/03/31,Punjab Kings,Punjab Kings,1
1173,1527678,2026/04/01,Delhi Capitals,Delhi Capitals,1


In [18]:
all_results = pd.concat([team1_results, team2_results]).sort_values("match_date")

print("All results shape:", all_results.shape)
print(all_results.head(10))

All results shape: (2350, 5)
   match_id  match_date                         team  \
0    335982  2008/04/18  Royal Challengers Bengaluru   
0    335982  2008/04/18        Kolkata Knight Riders   
1    335983  2008/04/19                 Punjab Kings   
2    335984  2008/04/19               Delhi Capitals   
2    335984  2008/04/19             Rajasthan Royals   
1    335983  2008/04/19          Chennai Super Kings   
4    335985  2008/04/20  Royal Challengers Bengaluru   
3    335986  2008/04/20          Sunrisers Hyderabad   
3    335986  2008/04/20        Kolkata Knight Riders   
4    335985  2008/04/20               Mumbai Indians   

                        winner  won  
0        Kolkata Knight Riders    0  
0        Kolkata Knight Riders    1  
1          Chennai Super Kings    0  
2               Delhi Capitals    1  
2               Delhi Capitals    0  
1          Chennai Super Kings    1  
4  Royal Challengers Bengaluru    1  
3        Kolkata Knight Riders    0  
3        Kol

In [19]:
all_results["recent_form"] = all_results.groupby("team")["won"].transform(
    lambda x: x.shift(1).rolling(window=5, min_periods=1).sum()
)

print("Recent form sample:")
print(all_results[["team", "match_date", "won", "recent_form"]].head(20))

Recent form sample:
                          team  match_date  won  recent_form
0  Royal Challengers Bengaluru  2008/04/18    0          NaN
0        Kolkata Knight Riders  2008/04/18    1          NaN
1                 Punjab Kings  2008/04/19    0          NaN
2               Delhi Capitals  2008/04/19    1          NaN
2             Rajasthan Royals  2008/04/19    0          NaN
1          Chennai Super Kings  2008/04/19    1          NaN
4  Royal Challengers Bengaluru  2008/04/20    1          0.0
3          Sunrisers Hyderabad  2008/04/20    0          NaN
3        Kolkata Knight Riders  2008/04/20    1          1.0
4               Mumbai Indians  2008/04/20    0          NaN
5             Rajasthan Royals  2008/04/21    1          0.0
5                 Punjab Kings  2008/04/21    0          0.0
6               Delhi Capitals  2008/04/22    1          1.0
6          Sunrisers Hyderabad  2008/04/22    0          0.0
7               Mumbai Indians  2008/04/23    0          0.0
7   

In [20]:
team1_form = all_results[["match_id", "team", "recent_form"]].copy()
team1_form.columns = ["match_id", "team1", "team1_recent_form"]
team1_form

,match_id,team1,team1_recent_form
0,335982,Royal Challengers Bengaluru,NaN
0,335982,Kolkata Knight Riders,NaN
1,335983,Punjab Kings,NaN
2,335984,Delhi Capitals,NaN
2,335984,Rajasthan Royals,NaN
...,...,...,...
1172,1527677,Gujarat Titans,2.0
1173,1527678,Delhi Capitals,1.0
1173,1527678,Lucknow Super Giants,1.0
1174,1527679,Sunrisers Hyderabad,3.0


In [21]:
team2_form = all_results[["match_id", "team", "recent_form"]].copy()
team2_form.columns = ["match_id", "team2", "team2_recent_form"]
team2_form

,match_id,team2,team2_recent_form
0,335982,Royal Challengers Bengaluru,NaN
0,335982,Kolkata Knight Riders,NaN
1,335983,Punjab Kings,NaN
2,335984,Delhi Capitals,NaN
2,335984,Rajasthan Royals,NaN
...,...,...,...
1172,1527677,Gujarat Titans,2.0
1173,1527678,Delhi Capitals,1.0
1173,1527678,Lucknow Super Giants,1.0
1174,1527679,Sunrisers Hyderabad,3.0


In [22]:
matches_clean = matches_clean.merge(team1_form, on=["match_id", "team1"], how="left")
matches_clean = matches_clean.merge(team2_form, on=["match_id", "team2"], how="left")
matches_clean

,match_id,season,match_date,team1,team2,venue,city,toss_winner,toss_decision,toss_winner_won,toss_decision_encoded,team1_win_rate,team2_win_rate,win_rate_diff,winner,result,target,team1_recent_form,team2_recent_form
0,335982,2007/08,2008/04/18,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bengaluru,field,0,1,0.49,0.51,-0.02,Kolkata Knight Riders,normal,0,NaN,NaN
1,335983,2007/08,2008/04/19,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,1,0,0.45,0.56,-0.11,Chennai Super Kings,normal,0,NaN,NaN
2,335984,2007/08,2008/04/19,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,0,0,0.44,0.49,-0.05,Delhi Capitals,normal,1,NaN,NaN
3,335986,2007/08,2008/04/20,Kolkata Knight Riders,Sunrisers Hyderabad,Eden Gardens,Kolkata,Sunrisers Hyderabad,bat,0,0,0.51,0.45,0.06,Kolkata Knight Riders,normal,1,1.0,NaN
4,335985,2007/08,2008/04/20,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai,Mumbai Indians,bat,0,0,0.55,0.49,0.06,Royal Challengers Bengaluru,normal,0,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1185,1527675,2026,2026/03/29,Kolkata Knight Riders,Mumbai Indians,"Wankhede Stadium, Mumbai",Mumbai,Mumbai Indians,field,1,1,0.51,0.55,-0.04,Mumbai Indians,normal,0,2.0,2.0
1186,1527676,2026,2026/03/30,Chennai Super Kings,Rajasthan Royals,"Barsapara Cricket Stadium, Guwahati",Guwahati,Rajasthan Royals,field,1,1,0.56,0.49,0.07,Rajasthan Royals,normal,0,2.0,2.0
1187,1527677,2026,2026/03/31,Gujarat Titans,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Punjab Kings,field,1,1,0.61,0.45,0.16,Punjab Kings,normal,0,2.0,2.0
1188,1527678,2026,2026/04/01,Lucknow Super Giants,Delhi Capitals,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Delhi Capitals,field,1,1,0.51,0.44,0.07,Delhi Capitals,normal,0,1.0,1.0


In [23]:
matches_clean["form_diff"] = matches_clean["team1_recent_form"] - matches_clean["team2_recent_form"]

In [24]:
print("Sample:")
print(matches_clean[["team1", "team2", "team1_recent_form", "team2_recent_form", "form_diff"]].head(10))

Sample:
                         team1                        team2  \
0  Royal Challengers Bengaluru        Kolkata Knight Riders   
1                 Punjab Kings          Chennai Super Kings   
2               Delhi Capitals             Rajasthan Royals   
3        Kolkata Knight Riders          Sunrisers Hyderabad   
4               Mumbai Indians  Royal Challengers Bengaluru   
5             Rajasthan Royals                 Punjab Kings   
6          Sunrisers Hyderabad               Delhi Capitals   
7          Chennai Super Kings               Mumbai Indians   
8          Sunrisers Hyderabad             Rajasthan Royals   
9                 Punjab Kings               Mumbai Indians   

   team1_recent_form  team2_recent_form  form_diff  
0                NaN                NaN        NaN  
1                NaN                NaN        NaN  
2                NaN                NaN        NaN  
3                1.0                NaN        NaN  
4                NaN             

In [25]:
# Head to Head record
h2h = matches_clean.groupby(["team1", "winner"])["match_id"].count().reset_index()
h2h.columns = ["team1", "winner", "h2h_wins"]
h2h

,team1,winner,h2h_wins
0,Chennai Super Kings,Chennai Super Kings,76
1,Chennai Super Kings,Defunct,1
2,Chennai Super Kings,Delhi Capitals,6
3,Chennai Super Kings,Gujarat Titans,3
4,Chennai Super Kings,Kolkata Knight Riders,6
...,...,...,...
106,Sunrisers Hyderabad,Mumbai Indians,10
107,Sunrisers Hyderabad,Punjab Kings,8
108,Sunrisers Hyderabad,Rajasthan Royals,8
109,Sunrisers Hyderabad,Royal Challengers Bengaluru,5


In [26]:
h2h = h2h[h2h["team1"] == h2h["winner"]][["team1", "h2h_wins"]]
h2h

,team1,h2h_wins
0,Chennai Super Kings,76
12,Defunct,37
22,Delhi Capitals,60
33,Gujarat Titans,15
44,Kolkata Knight Riders,67
54,Lucknow Super Giants,20
65,Mumbai Indians,74
76,Punjab Kings,59
86,Rajasthan Royals,57
98,Royal Challengers Bengaluru,73


In [27]:
matches_clean = matches_clean.merge(h2h, on="team1", how="left")
matches_clean = matches_clean.rename(columns={"h2h_wins": "team1_h2h_wins"})
matches_clean["team1_h2h_wins"] = matches_clean["team1_h2h_wins"].fillna(0)
matches_clean

,match_id,season,match_date,team1,team2,venue,city,toss_winner,toss_decision,toss_winner_won,...,team1_win_rate,team2_win_rate,win_rate_diff,winner,result,target,team1_recent_form,team2_recent_form,form_diff,team1_h2h_wins
0,335982,2007/08,2008/04/18,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bengaluru,field,0,...,0.49,0.51,-0.02,Kolkata Knight Riders,normal,0,NaN,NaN,NaN,73
1,335983,2007/08,2008/04/19,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,1,...,0.45,0.56,-0.11,Chennai Super Kings,normal,0,NaN,NaN,NaN,59
2,335984,2007/08,2008/04/19,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,0,...,0.44,0.49,-0.05,Delhi Capitals,normal,1,NaN,NaN,NaN,60
3,335986,2007/08,2008/04/20,Kolkata Knight Riders,Sunrisers Hyderabad,Eden Gardens,Kolkata,Sunrisers Hyderabad,bat,0,...,0.51,0.45,0.06,Kolkata Knight Riders,normal,1,1.0,NaN,NaN,67
4,335985,2007/08,2008/04/20,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai,Mumbai Indians,bat,0,...,0.55,0.49,0.06,Royal Challengers Bengaluru,normal,0,NaN,0.0,NaN,74
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1185,1527675,2026,2026/03/29,Kolkata Knight Riders,Mumbai Indians,"Wankhede Stadium, Mumbai",Mumbai,Mumbai Indians,field,1,...,0.51,0.55,-0.04,Mumbai Indians,normal,0,2.0,2.0,0.0,67
1186,1527676,2026,2026/03/30,Chennai Super Kings,Rajasthan Royals,"Barsapara Cricket Stadium, Guwahati",Guwahati,Rajasthan Royals,field,1,...,0.56,0.49,0.07,Rajasthan Royals,normal,0,2.0,2.0,0.0,76
1187,1527677,2026,2026/03/31,Gujarat Titans,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Punjab Kings,field,1,...,0.61,0.45,0.16,Punjab Kings,normal,0,2.0,2.0,0.0,15
1188,1527678,2026,2026/04/01,Lucknow Super Giants,Delhi Capitals,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Delhi Capitals,field,1,...,0.51,0.44,0.07,Delhi Capitals,normal,0,1.0,1.0,0.0,20


In [28]:
print("H2H sample:")
print(matches_clean[["team1", "team2", "team1_h2h_wins"]].head(10))

H2H sample:
                         team1                        team2  team1_h2h_wins
0  Royal Challengers Bengaluru        Kolkata Knight Riders              73
1                 Punjab Kings          Chennai Super Kings              59
2               Delhi Capitals             Rajasthan Royals              60
3        Kolkata Knight Riders          Sunrisers Hyderabad              67
4               Mumbai Indians  Royal Challengers Bengaluru              74
5             Rajasthan Royals                 Punjab Kings              57
6          Sunrisers Hyderabad               Delhi Capitals              60
7          Chennai Super Kings               Mumbai Indians              76
8          Sunrisers Hyderabad             Rajasthan Royals              60
9                 Punjab Kings               Mumbai Indians              59


In [29]:
# Venue win rate
venue_played_t1 = matches_clean.groupby(["venue", "team1"])["match_id"].count().reset_index()
venue_played_t1.columns = ["venue", "team", "played"]
venue_played_t1

,venue,team,played
0,Arun Jaitley Stadium,Delhi Capitals,9
1,Arun Jaitley Stadium,Kolkata Knight Riders,1
2,Arun Jaitley Stadium,Mumbai Indians,1
3,Arun Jaitley Stadium,Punjab Kings,2
4,Arun Jaitley Stadium,Rajasthan Royals,1
...,...,...,...
281,"Zayed Cricket Stadium, Abu Dhabi",Kolkata Knight Riders,1
282,"Zayed Cricket Stadium, Abu Dhabi",Mumbai Indians,2
283,"Zayed Cricket Stadium, Abu Dhabi",Punjab Kings,1
284,"Zayed Cricket Stadium, Abu Dhabi",Royal Challengers Bengaluru,1


In [30]:
venue_played_t2 = matches_clean.groupby(["venue", "team2"])["match_id"].count().reset_index()
venue_played_t2.columns = ["venue", "team", "played"]
venue_played_t2

,venue,team,played
0,Arun Jaitley Stadium,Chennai Super Kings,2
1,Arun Jaitley Stadium,Delhi Capitals,5
2,Arun Jaitley Stadium,Kolkata Knight Riders,1
3,Arun Jaitley Stadium,Mumbai Indians,1
4,Arun Jaitley Stadium,Rajasthan Royals,1
...,...,...,...
386,"Zayed Cricket Stadium, Abu Dhabi",Kolkata Knight Riders,2
387,"Zayed Cricket Stadium, Abu Dhabi",Mumbai Indians,1
388,"Zayed Cricket Stadium, Abu Dhabi",Rajasthan Royals,2
389,"Zayed Cricket Stadium, Abu Dhabi",Royal Challengers Bengaluru,1


In [31]:
venue_played = pd.concat([venue_played_t1, venue_played_t2])
venue_played = venue_played.groupby(["venue", "team"])["played"].sum().reset_index()
venue_played

,venue,team,played
0,Arun Jaitley Stadium,Chennai Super Kings,2
1,Arun Jaitley Stadium,Delhi Capitals,14
2,Arun Jaitley Stadium,Kolkata Knight Riders,2
3,Arun Jaitley Stadium,Mumbai Indians,2
4,Arun Jaitley Stadium,Punjab Kings,2
...,...,...,...
454,"Zayed Cricket Stadium, Abu Dhabi",Mumbai Indians,3
455,"Zayed Cricket Stadium, Abu Dhabi",Punjab Kings,1
456,"Zayed Cricket Stadium, Abu Dhabi",Rajasthan Royals,2
457,"Zayed Cricket Stadium, Abu Dhabi",Royal Challengers Bengaluru,2


In [32]:
venue_wins = matches_clean.groupby(["venue", "winner"])["match_id"].count().reset_index()
venue_wins.columns = ["venue", "team", "wins"]
venue_wins

,venue,team,wins
0,Arun Jaitley Stadium,Chennai Super Kings,1
1,Arun Jaitley Stadium,Delhi Capitals,7
2,Arun Jaitley Stadium,Mumbai Indians,1
3,Arun Jaitley Stadium,Punjab Kings,1
4,Arun Jaitley Stadium,Royal Challengers Bengaluru,1
...,...,...,...
362,"Zayed Cricket Stadium, Abu Dhabi",Delhi Capitals,1
363,"Zayed Cricket Stadium, Abu Dhabi",Kolkata Knight Riders,2
364,"Zayed Cricket Stadium, Abu Dhabi",Mumbai Indians,2
365,"Zayed Cricket Stadium, Abu Dhabi",Rajasthan Royals,1


In [33]:
venue_stats = venue_played.merge(venue_wins, on=["venue", "team"], how="left")
venue_stats["wins"] = venue_stats["wins"].fillna(0)
venue_stats["venue_win_rate"] = round(venue_stats["wins"] / venue_stats["played"], 2)
venue_stats

,venue,team,played,wins,venue_win_rate
0,Arun Jaitley Stadium,Chennai Super Kings,2,1.0,0.50
1,Arun Jaitley Stadium,Delhi Capitals,14,7.0,0.50
2,Arun Jaitley Stadium,Kolkata Knight Riders,2,0.0,0.00
3,Arun Jaitley Stadium,Mumbai Indians,2,1.0,0.50
4,Arun Jaitley Stadium,Punjab Kings,2,1.0,0.50
...,...,...,...,...,...
454,"Zayed Cricket Stadium, Abu Dhabi",Mumbai Indians,3,2.0,0.67
455,"Zayed Cricket Stadium, Abu Dhabi",Punjab Kings,1,0.0,0.00
456,"Zayed Cricket Stadium, Abu Dhabi",Rajasthan Royals,2,1.0,0.50
457,"Zayed Cricket Stadium, Abu Dhabi",Royal Challengers Bengaluru,2,0.0,0.00


In [34]:
print("Venue stats sample:")
print(venue_stats.head(10))

Venue stats sample:
                         venue                         team  played  wins  \
0         Arun Jaitley Stadium          Chennai Super Kings       2   1.0   
1         Arun Jaitley Stadium               Delhi Capitals      14   7.0   
2         Arun Jaitley Stadium        Kolkata Knight Riders       2   0.0   
3         Arun Jaitley Stadium               Mumbai Indians       2   1.0   
4         Arun Jaitley Stadium                 Punjab Kings       2   1.0   
5         Arun Jaitley Stadium             Rajasthan Royals       2   0.0   
6         Arun Jaitley Stadium  Royal Challengers Bengaluru       2   1.0   
7         Arun Jaitley Stadium          Sunrisers Hyderabad       2   2.0   
8  Arun Jaitley Stadium, Delhi          Chennai Super Kings       4   2.0   
9  Arun Jaitley Stadium, Delhi               Delhi Capitals      17   6.0   

   venue_win_rate  
0            0.50  
1            0.50  
2            0.00  
3            0.50  
4            0.50  
5           

In [35]:
venue_t1 = venue_stats[["venue", "team", "venue_win_rate"]].copy()
venue_t1.columns = ["venue", "team1", "team1_venue_win_rate"]

venue_t2 = venue_stats[["venue", "team", "venue_win_rate"]].copy()
venue_t2.columns = ["venue", "team2", "team2_venue_win_rate"]


In [36]:
matches_clean = matches_clean.merge(venue_t1, on=["venue", "team1"], how="left")
matches_clean = matches_clean.merge(venue_t2, on=["venue", "team2"], how="left")
matches_clean

,match_id,season,match_date,team1,team2,venue,city,toss_winner,toss_decision,toss_winner_won,...,win_rate_diff,winner,result,target,team1_recent_form,team2_recent_form,form_diff,team1_h2h_wins,team1_venue_win_rate,team2_venue_win_rate
0,335982,2007/08,2008/04/18,Royal Challengers Bengaluru,Kolkata Knight Riders,M Chinnaswamy Stadium,Bangalore,Royal Challengers Bengaluru,field,0,...,-0.02,Kolkata Knight Riders,normal,0,NaN,NaN,NaN,73,0.46,0.55
1,335983,2007/08,2008/04/19,Punjab Kings,Chennai Super Kings,"Punjab Cricket Association Stadium, Mohali",Chandigarh,Chennai Super Kings,bat,1,...,-0.11,Chennai Super Kings,normal,0,NaN,NaN,NaN,59,0.51,0.75
2,335984,2007/08,2008/04/19,Delhi Capitals,Rajasthan Royals,Feroz Shah Kotla,Delhi,Rajasthan Royals,bat,0,...,-0.05,Delhi Capitals,normal,1,NaN,NaN,NaN,60,0.41,0.57
3,335986,2007/08,2008/04/20,Kolkata Knight Riders,Sunrisers Hyderabad,Eden Gardens,Kolkata,Sunrisers Hyderabad,bat,0,...,0.06,Kolkata Knight Riders,normal,1,1.0,NaN,NaN,67,0.61,0.18
4,335985,2007/08,2008/04/20,Mumbai Indians,Royal Challengers Bengaluru,Wankhede Stadium,Mumbai,Mumbai Indians,bat,0,...,0.06,Royal Challengers Bengaluru,normal,0,NaN,0.0,NaN,74,0.61,0.30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1185,1527675,2026,2026/03/29,Kolkata Knight Riders,Mumbai Indians,"Wankhede Stadium, Mumbai",Mumbai,Mumbai Indians,field,1,...,-0.04,Mumbai Indians,normal,0,2.0,2.0,0.0,67,0.40,0.62
1186,1527676,2026,2026/03/30,Chennai Super Kings,Rajasthan Royals,"Barsapara Cricket Stadium, Guwahati",Guwahati,Rajasthan Royals,field,1,...,0.07,Rajasthan Royals,normal,0,2.0,2.0,0.0,76,0.00,0.50
1187,1527677,2026,2026/03/31,Gujarat Titans,Punjab Kings,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Punjab Kings,field,1,...,0.16,Punjab Kings,normal,0,2.0,2.0,0.0,15,0.00,1.00
1188,1527678,2026,2026/04/01,Lucknow Super Giants,Delhi Capitals,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Delhi Capitals,field,1,...,0.07,Delhi Capitals,normal,0,1.0,1.0,0.0,20,0.41,0.75


In [37]:
matches_clean["venue_win_rate_diff"] = matches_clean["team1_venue_win_rate"] - matches_clean["team2_venue_win_rate"]

print("Shape after all new features:", matches_clean.shape)
print(matches_clean[["team1", "team2", "venue", "team1_venue_win_rate", "team2_venue_win_rate"]].head(10))

Shape after all new features: (1190, 24)
                         team1                        team2  \
0  Royal Challengers Bengaluru        Kolkata Knight Riders   
1                 Punjab Kings          Chennai Super Kings   
2               Delhi Capitals             Rajasthan Royals   
3        Kolkata Knight Riders          Sunrisers Hyderabad   
4               Mumbai Indians  Royal Challengers Bengaluru   
5             Rajasthan Royals                 Punjab Kings   
6          Sunrisers Hyderabad               Delhi Capitals   
7          Chennai Super Kings               Mumbai Indians   
8          Sunrisers Hyderabad             Rajasthan Royals   
9                 Punjab Kings               Mumbai Indians   

                                        venue  team1_venue_win_rate  \
0                       M Chinnaswamy Stadium                  0.46   
1  Punjab Cricket Association Stadium, Mohali                  0.51   
2                            Feroz Shah Kotla       

In [38]:
matches_clean.to_csv(FEATURES_CSV, index=False)
print("Saved! Shape:", matches_clean.shape)

Saved! Shape: (1190, 24)


In [39]:
matches_clean = matches_clean.drop_duplicates("match_id")
print("Shape after removing duplicates:", matches_clean.shape)

Shape after removing duplicates: (1175, 24)


In [40]:
X = matches_clean[[
    "toss_winner_won",
    "toss_decision_encoded",
    "team1_win_rate",
    "team2_win_rate",
    "win_rate_diff",
    "team1_recent_form",    # NEW
    "team2_recent_form",    # NEW
    "form_diff",            # NEW
    "team1_h2h_wins",       # NEW
    "team1_venue_win_rate", # NEW
    "team2_venue_win_rate", # NEW
    "venue_win_rate_diff"   # NEW
]]

y = matches_clean["target"]

In [41]:
X = X.dropna()
y = y[X.index]

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows :", len(X_test))

Training rows: 817
Testing rows : 351


In [43]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)
lr_accuracy = accuracy_score(y_test, lr_model.predict(X_test))

# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))

# XGBoost
xgb_model = XGBClassifier(random_state=42, eval_metric="logloss")
xgb_model.fit(X_train, y_train)
xgb_accuracy = accuracy_score(y_test, xgb_model.predict(X_test))

# Comparison table
results = pd.DataFrame({
    "Model"   : ["Logistic Regression", "Random Forest", "XGBoost"],
    "Old Accuracy" : [62.61, 69.97, 72.24],
    "New Accuracy" : [
        round(lr_accuracy * 100, 2),
        round(rf_accuracy * 100, 2),
        round(xgb_accuracy * 100, 2)
    ]
})

results["Improvement"] = results["New Accuracy"] - results["Old Accuracy"]
results = results.sort_values("New Accuracy", ascending=False).reset_index(drop=True)
print(results)

D:\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                 Model  Old Accuracy  New Accuracy  Improvement
0        Random Forest         69.97         76.64         6.67
1              XGBoost         72.24         75.50         3.26
2  Logistic Regression         62.61         74.07        11.46


In [44]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_accuracy = accuracy_score(y_test, lr_model.predict(X_test))
print("Logistic Regression Accuracy:", round(lr_accuracy * 100, 2), "%")

Logistic Regression Accuracy: 74.36 %


### Random Forest is now the best model — the new features (recent form, head to head, venue win rate) gave it the biggest boost.

In [45]:
# Saving the trained model

import joblib
MODEL_PATH = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\rf_model.pkl"

joblib.dump(rf_model, MODEL_PATH)
print("Model saved successfully!")

Model saved successfully!


In [46]:
# Verify if the model loads correctly

loaded_model = joblib.load(MODEL_PATH)
test_accuracy = accuracy_score(y_test, loaded_model.predict(X_test))
print("Loaded model accuracy:", round(test_accuracy * 100, 2), "%")

Loaded model accuracy: 76.64 %


In [47]:
active_teams = [
    "Mumbai Indians",
    "Chennai Super Kings",
    "Royal Challengers Bengaluru",
    "Kolkata Knight Riders",
    "Delhi Capitals",
    "Punjab Kings",
    "Rajasthan Royals",
    "Sunrisers Hyderabad",
    "Gujarat Titans",
    "Lucknow Super Giants"
]

In [49]:
team_stats = matches_clean.groupby("team1").agg(
    win_rate     = ("team1_win_rate", "last"),
    recent_form  = ("team1_recent_form", "last"),
    venue_wr     = ("team1_venue_win_rate", "last")
).reset_index()

team_stats.columns = ["team", "win_rate", "recent_form", "venue_wr"]
team_stats

,team,win_rate,recent_form,venue_wr
0,Chennai Super Kings,0.56,2.0,0.00
1,Defunct,0.38,2.0,0.46
2,Delhi Capitals,0.44,1.0,0.35
3,Gujarat Titans,0.61,2.0,0.00
4,Kolkata Knight Riders,0.51,2.0,0.40
5,Lucknow Super Giants,0.51,1.0,0.41
6,Mumbai Indians,0.55,3.0,0.00
7,Punjab Kings,0.45,3.0,0.30
8,Rajasthan Royals,0.49,2.0,0.40
9,Royal Challengers Bengaluru,0.49,4.0,0.50


In [50]:
team_stats = team_stats[team_stats["team"].isin(active_teams)]
team_stats

,team,win_rate,recent_form,venue_wr
0,Chennai Super Kings,0.56,2.0,0.00
2,Delhi Capitals,0.44,1.0,0.35
3,Gujarat Titans,0.61,2.0,0.00
4,Kolkata Knight Riders,0.51,2.0,0.40
5,Lucknow Super Giants,0.51,1.0,0.41
6,Mumbai Indians,0.55,3.0,0.00
7,Punjab Kings,0.45,3.0,0.30
8,Rajasthan Royals,0.49,2.0,0.40
9,Royal Challengers Bengaluru,0.49,4.0,0.50
10,Sunrisers Hyderabad,0.45,3.0,0.50


In [51]:
print(team_stats.sort_values("win_rate", ascending=False).to_string(index=False))

                       team  win_rate  recent_form  venue_wr
             Gujarat Titans      0.61          2.0      0.00
        Chennai Super Kings      0.56          2.0      0.00
             Mumbai Indians      0.55          3.0      0.00
      Kolkata Knight Riders      0.51          2.0      0.40
       Lucknow Super Giants      0.51          1.0      0.41
           Rajasthan Royals      0.49          2.0      0.40
Royal Challengers Bengaluru      0.49          4.0      0.50
               Punjab Kings      0.45          3.0      0.30
        Sunrisers Hyderabad      0.45          3.0      0.50
             Delhi Capitals      0.44          1.0      0.35


In [52]:
#  Predicting every possible matchup

from itertools import combinations

# Generate all possible match combinations between 10 teams
matchups = list(combinations(active_teams, 2))
print(f"Total possible matchups: {len(matchups)}")

# Build a dataframe of all matchups
matchup_df = pd.DataFrame(matchups, columns=["team1", "team2"])

# Attach team1 stats
matchup_df = matchup_df.merge(
    team_stats.rename(columns={
        "team"      : "team1",
        "win_rate"  : "team1_win_rate",
        "recent_form": "team1_recent_form",
        "venue_wr"  : "team1_venue_win_rate"
    }),
    on="team1", how="left"
)

# Attach team2 stats
matchup_df = matchup_df.merge(
    team_stats.rename(columns={
        "team"      : "team2",
        "win_rate"  : "team2_win_rate",
        "recent_form": "team2_recent_form",
        "venue_wr"  : "team2_venue_win_rate"
    }),
    on="team2", how="left"
)

print("Matchup dataframe shape:", matchup_df.shape)
print(matchup_df.head(5).to_string())

Total possible matchups: 45
Matchup dataframe shape: (45, 8)
            team1                        team2  team1_win_rate  team1_recent_form  team1_venue_win_rate  team2_win_rate  team2_recent_form  team2_venue_win_rate
0  Mumbai Indians          Chennai Super Kings            0.55                3.0                   0.0            0.56                2.0                  0.00
1  Mumbai Indians  Royal Challengers Bengaluru            0.55                3.0                   0.0            0.49                4.0                  0.50
2  Mumbai Indians        Kolkata Knight Riders            0.55                3.0                   0.0            0.51                2.0                  0.40
3  Mumbai Indians               Delhi Capitals            0.55                3.0                   0.0            0.44                1.0                  0.35
4  Mumbai Indians                 Punjab Kings            0.55                3.0                   0.0            0.45               

In [53]:
# Filling remaining features and predicting


matchup_df["toss_winner_won"]       = 0   # neutral — we don't know toss yet
matchup_df["toss_decision_encoded"] = 1   # most teams choose to field in 2026
matchup_df["win_rate_diff"]         = matchup_df["team1_win_rate"] - matchup_df["team2_win_rate"]
matchup_df["form_diff"]             = matchup_df["team1_recent_form"] - matchup_df["team2_recent_form"]
matchup_df["team1_h2h_wins"]        = 5   # neutral average
matchup_df["venue_win_rate_diff"]   = matchup_df["team1_venue_win_rate"] - matchup_df["team2_venue_win_rate"]

# Selecting same features the model was trained on
feature_cols = [
    "toss_winner_won",
    "toss_decision_encoded",
    "team1_win_rate",
    "team2_win_rate",
    "win_rate_diff",
    "team1_recent_form",
    "team2_recent_form",
    "form_diff",
    "team1_h2h_wins",
    "team1_venue_win_rate",
    "team2_venue_win_rate",
    "venue_win_rate_diff"
]

# Filling any remaining NaN with 0
matchup_df[feature_cols] = matchup_df[feature_cols].fillna(0)

# Geting win probability for each matchup
# predict_proba gives probability of winning (0 to 1)
matchup_df["team1_win_prob"] = loaded_model.predict_proba(
    matchup_df[feature_cols]
)[:, 1]

matchup_df["team2_win_prob"] = 1 - matchup_df["team1_win_prob"]

print(matchup_df[["team1", "team2", "team1_win_prob", "team2_win_prob"]].head(10).to_string())

                 team1                        team2  team1_win_prob  team2_win_prob
0       Mumbai Indians          Chennai Super Kings            0.51            0.49
1       Mumbai Indians  Royal Challengers Bengaluru            0.20            0.80
2       Mumbai Indians        Kolkata Knight Riders            0.25            0.75
3       Mumbai Indians               Delhi Capitals            0.39            0.61
4       Mumbai Indians                 Punjab Kings            0.46            0.54
5       Mumbai Indians             Rajasthan Royals            0.26            0.74
6       Mumbai Indians          Sunrisers Hyderabad            0.31            0.69
7       Mumbai Indians               Gujarat Titans            0.50            0.50
8       Mumbai Indians         Lucknow Super Giants            0.25            0.75
9  Chennai Super Kings  Royal Challengers Bengaluru            0.27            0.73


In [54]:
# Calculating overall win probability per team

# For each team, averaging their win probability across all matchups
team1_probs = matchup_df[["team1", "team1_win_prob"]].rename(
    columns={"team1": "team", "team1_win_prob": "win_prob"}
)

team2_probs = matchup_df[["team2", "team2_win_prob"]].rename(
    columns={"team2": "team", "team2_win_prob": "win_prob"}
)

all_probs = pd.concat([team1_probs, team2_probs])

# Average win probability per team
final_prediction = all_probs.groupby("team")["win_prob"].mean().reset_index()
final_prediction.columns = ["team", "win_probability"]

# Converting to percentage and sorting
final_prediction["win_probability_%"] = round(
    final_prediction["win_probability"] * 100, 2
)
final_prediction = final_prediction.sort_values(
    "win_probability_%", ascending=False
).reset_index(drop=True)

# Adding rank
final_prediction.index = final_prediction.index + 1
final_prediction.index.name = "rank"

print("\n🏆 IPL 2026 WINNER PREDICTION 🏆\n")
print(final_prediction[["team", "win_probability_%"]].to_string())


🏆 IPL 2026 WINNER PREDICTION 🏆

                             team  win_probability_%
rank                                                
1     Royal Challengers Bengaluru              75.28
2           Kolkata Knight Riders              64.89
3                Rajasthan Royals              59.11
4             Sunrisers Hyderabad              56.22
5                  Delhi Capitals              54.89
6                    Punjab Kings              52.06
7            Lucknow Super Giants              42.56
8             Chennai Super Kings              37.00
9                  Mumbai Indians              34.78
10                 Gujarat Titans              23.22


In [55]:
PREDICTIONS_CSV = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_2026_predictions.csv"

final_prediction.to_csv(PREDICTIONS_CSV)
print("Predictions saved to:", PREDICTIONS_CSV)

Predictions saved to: C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_2026_predictions.csv


### Why RCB on top makes complete sense:

##### They are the defending champions (won IPL 2025)
##### Their recent form and win rate are the highest in the dataset
##### The model learned that defending champions carry strong momentum.

### Why MI and CSK are low:

##### Despite being the most successful teams historically, their recent form dragged them down
##### This is actually a good sign — the model is picking up current form over just historical reputation

## Final model: Random Forest — 76.64% accuracy



In [59]:
# Loading predictions
PREDICTIONS_CSV = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_2026_predictions.csv"
predictions = pd.read_csv(PREDICTIONS_CSV)

# Cleaning column names
predictions.columns = ["Rank", "Team", "Win_Probability", "Win_Probability_Pct"]

# Adding one helper column — top 4 are playoff qualifiers
predictions["Qualifier"] = predictions["Rank"].apply(
    lambda x: "Yes" if x <= 4 else "No"
)

print(predictions)

# Save
PBI_CSV = r"C:\Users\A_R COMPUTERS\Desktop\Vineet\Project\IPL Prediction\ipl_2026_powerbi.csv"
predictions.to_csv(PBI_CSV, index=False)
print("File saved!")

   Rank                         Team  Win_Probability  Win_Probability_Pct  \
0     1  Royal Challengers Bengaluru         0.752778                75.28   
1     2        Kolkata Knight Riders         0.648889                64.89   
2     3             Rajasthan Royals         0.591111                59.11   
3     4          Sunrisers Hyderabad         0.562222                56.22   
4     5               Delhi Capitals         0.548889                54.89   
5     6                 Punjab Kings         0.520556                52.06   
6     7         Lucknow Super Giants         0.425556                42.56   
7     8          Chennai Super Kings         0.370000                37.00   
8     9               Mumbai Indians         0.347778                34.78   
9    10               Gujarat Titans         0.232222                23.22   

  Qualifier  
0       Yes  
1       Yes  
2       Yes  
3       Yes  
4        No  
5        No  
6        No  
7        No  
8        No  
9